In [1]:
!pip install wandb

In [2]:
import wandb
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from kaggle_secrets import UserSecretsClient

class Net(nn.Module):
    def __init__(self, hidden_size, dropout, activation):
        super().__init__()
        self.fc1 = nn.Linear(32*32*3, hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.act = getattr(torch.nn, activation)()
        self.fc2 = nn.Linear(hidden_size, 10)
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.act(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

def train(model, loader, optimizer, criterion, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for data, targets in loader:
        data, targets = data.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(data)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * data.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(targets).sum().item()
        total += targets.size(0)
    return running_loss / total, correct / total

def validate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for data, targets in loader:
            data, targets = data.to(device), targets.to(device)
            outputs = model(data)
            loss = criterion(outputs, targets)
            running_loss += loss.item() * data.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(targets).sum().item()
            total += targets.size(0)
    return running_loss / total, correct / total

Необходимо предварительно создать аккаунт на W&B и авторизоваться через сгенерированный ключ.
В kaggle для хранения данных авторизации удобно пользоваться аддоном Secrets, в котором можно хранить именованные ключи. В коде ниже происходит авторизация по ключу `wandb_api_key`. Получить API key можно в профиле W&B.

In [3]:
user_secrets = UserSecretsClient()
my_secret = user_secrets.get_secret("wandb_api_key") 
wandb.login(key=my_secret)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ladaogneva13 (ladaogneva13-bauman-moscow-state-technical-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
config = {
    'epochs': 10, 'lr': 0.001, 'batch_size': 64,
    'hidden_size': 128, 'dropout': 0.5, 'activation': 'ReLU'
}
wandb.init(project="cifar10_classification", config=config)
transform = transforms.ToTensor()
train_set = datasets.CIFAR10(root='.', train=True, download=True, transform=transform)
val_set = datasets.CIFAR10(root='.', train=False, download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=config['batch_size'], shuffle=True)
val_loader = DataLoader(val_set, batch_size=config['batch_size'])

model = Net(config['hidden_size'], config['dropout'], config['activation']).to(device)
optimizer = optim.Adam(model.parameters(), lr=config['lr'])
criterion = nn.CrossEntropyLoss()

for epoch in range(config['epochs']):
    loss, acc = train(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    wandb.log({'epoch': epoch, 'train_loss': loss, 'train_acc': acc, 'val_loss': val_loss, 'val_acc': val_acc})
    print(f'Epoch {epoch+1}: train loss={loss:.4f}, train acc={acc:.4f}; val loss={val_loss:.4f}, val acc={val_acc:.4f}')

100%|██████████| 170M/170M [00:11<00:00, 14.4MB/s] 


Epoch 1: train loss=2.0625, train acc=0.2333; val loss=1.8780, val acc=0.3240
Epoch 2: train loss=1.9932, train acc=0.2581; val loss=1.8817, val acc=0.3374
Epoch 3: train loss=1.9734, train acc=0.2711; val loss=1.8278, val acc=0.3527
Epoch 4: train loss=1.9502, train acc=0.2801; val loss=1.8191, val acc=0.3598
Epoch 5: train loss=1.9369, train acc=0.2844; val loss=1.7794, val acc=0.3665
Epoch 6: train loss=1.9343, train acc=0.2835; val loss=1.8165, val acc=0.3597
Epoch 7: train loss=1.9276, train acc=0.2874; val loss=1.7929, val acc=0.3544
Epoch 8: train loss=1.9176, train acc=0.2913; val loss=1.7842, val acc=0.3570
Epoch 9: train loss=1.9117, train acc=0.2929; val loss=1.7999, val acc=0.3613
Epoch 10: train loss=1.9124, train acc=0.2946; val loss=1.7784, val acc=0.3776
